# 레슨 12 - 에이전트 스크래치패드를 통한 채팅 기록 축소

이 노트북은 Microsoft Agent Framework를 사용하여 긴 대화에서 컨텍스트를 관리하는 방법을 보여줍니다. 대화가 길어지면서 토큰 수가 증가하여 결국 모델의 컨텍스트 창을 초과하게 됩니다. 우리는 이를 <strong>컨텍스트 요약 패턴</strong>과 지속적인 메모리를 위한 <strong>에이전트 스크래치패드</strong>로 해결합니다.

## 배울 내용:
1. **컨텍스트 관리가 중요한 이유**: 토큰 제한과 컨텍스트 창 이해하기
2. **컨텍스트 인식 에이전트**: 자신의 대화 컨텍스트를 관리하는 에이전트 구축
3. **컨텍스트 요약 패턴**: 도구를 사용하여 대화 기록을 압축하기
4. **에이전트 스크래치패드**: 컨텍스트 축소를 견디는 지속적인 메모리

## 전제 조건:
- 환경 변수가 구성된 Azure OpenAI 설정
- 이전 레슨에서 배운 기본 에이전트 개념 이해


## 설정


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv --quiet

In [ ]:
import os
import asyncio
import dotenv
from datetime import datetime
from pathlib import Path

from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

In [ ]:
dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

print("Microsoft Foundry client configured")

## 컨텍스트 관리가 중요한 이유

모든 LLM은 유한한 <strong>컨텍스트 창</strong>을 가지고 있습니다 — 단일 요청에서 처리할 수 있는 최대 토큰 수입니다. 다회차 대화가 진행됨에 따라:

- <strong>토큰 수가 각 사용자 메시지와 도우미 응답마다 선형적으로 증가</strong>합니다.
- <strong>프롬프트 토큰이 비용을 지배</strong>하며, 전체 이력이 매 턴마다 재전송됩니다.
- 결국 대화가 <strong>컨텍스트 창을 초과</strong>하면 모델이 잘라내거나 오류를 발생시킵니다.

### 컨텍스트 관리 전략

| 전략 | 작동 방식 | 트레이드오프 |
|---|---|---|
| <strong>잘라내기</strong> | 가장 오래된 메시지 삭제 | 초기 문맥 손실 |
| <strong>요약</strong> | 오래된 메시지를 요약으로 압축 | 일부 세부정보 손실, 하지만 핵심 내용 유지 |
| **스크래치패드 / 외부 메모리** | 대화 외부에 주요 사실 저장 | 도구 호출 필요하지만 어떤 축소에도 유지됨 |

이 노트북에서는 <strong>요약</strong>과 <strong>스크래치패드 도구</strong>를 결합하여 대화 기록이 압축되어도 에이전트가 연속성을 유지할 수 있도록 합니다.


## 컨텍스트 인식 에이전트 만들기


In [ ]:
agent = client.as_agent(
    name="ContextAwareAgent",
    instructions="""You are a helpful travel planning assistant with excellent memory management.
When conversations get long:
1. Summarize previous context into key points
2. Track user preferences mentioned earlier
3. Reference previous decisions without repeating full details
Always maintain continuity while being concise.""",
)

print("Context-aware travel planning agent created")

## 긴 대화 시뮬레이션

여러 턴으로 이루어진 대화를 살펴보면서 어떻게 문맥이 누적되는지 알아봅시다. 에이전트는 각 턴마다 주요 세부 사항(선호도, 예산, 여행 날짜)을 유지하고 연속성을 보여줘야 합니다.


In [ ]:
session = agent.create_session()

# Turn 1 - Initial preferences
response = await agent.run("I'm planning a trip to Japan. I love sushi, temples, and photography.", session=session)
print(f"Turn 1: {response}\n")

# Turn 2 - More details
response = await agent.run("My budget is $3000 and I'll be traveling solo for 10 days in April.", session=session)
print(f"Turn 2: {response}\n")

# Turn 3 - Test context retention
response = await agent.run("Based on everything I've told you so far, what's the one thing you'd recommend I not miss?", session=session)
print(f"Turn 3: {response}\n")

에이전트가 이전 대화에서의 맥락을 어떻게 유지하는지 주목하세요 — 일본, 초밥, 사원, 사진 촬영, 3000달러 예산, 솔로 여행, 4월 일정에 대해 알고 있습니다. 짧은 대화에서는 잘 작동하지만, 대화가 길어질수록 전체 이력을 다시 보내는 비용이 커집니다.

맥락 누적을 보기 위해 대화를 더 이어가 보겠습니다:


In [ ]:
# Turn 4 - Expand the conversation
response = await agent.run("What about accommodation? I prefer traditional Japanese inns.", session=session)
print(f"Turn 4: {response}\n")

# Turn 5 - Change of plans
response = await agent.run("Actually, I've changed my mind about the dates. I'll go in October instead for the autumn colors.", session=session)
print(f"Turn 5: {response}\n")

# Turn 6 - Test retention after change
response = await agent.run("Summarize my complete travel plan so far — destination, budget, duration, interests, accommodation, and timing.", session=session)
print(f"Turn 6: {response}\n")

## 컨텍스트 요약 패턴

대화가 진행됨에 따라, 축적된 컨텍스트를 간결한 형식으로 압축하기 위해 <strong>요약 도구</strong>를 사용할 수 있습니다. 에이전트는 이 도구를 호출하여 주요 선호도를 기록하므로, 이전 메시지가 삭제되더라도 핵심 정보는 보존됩니다.

이 패턴은 더 정교한 기록 축소를 위한 기본 구조입니다:
1. 에이전트가 대화에서 핵심 사실을 식별합니다
2. 요약 도구를 호출하여 이를 저장합니다
3. 요약에 중요한 내용이 담겨 있으므로 이전 메시지는 안전하게 제거할 수 있습니다

아래에서는 에이전트가 학습한 내용을 간결하게 요약해 기록할 수 있는 `summarize_preferences` 도구를 정의합니다.


In [ ]:
@tool(approval_mode="never_require")
def summarize_preferences(conversation_notes: str) -> str:
    """Summarize accumulated user preferences into a compact format."""
    return f"[SUMMARY] User preferences recorded: {conversation_notes}"


# Create an enhanced agent with the summarization tool
summarizing_agent = client.as_agent(
    name="SummarizingTravelAgent",
    instructions="""You are a helpful travel planning assistant that actively manages conversation context.

CONTEXT MANAGEMENT RULES:
1. After gathering several user preferences, call summarize_preferences() to record a compact summary
2. When the user asks you to recall details, reference your recorded summaries
3. Keep responses concise — avoid restating the entire history

PLANNING PROCESS:
1. Gather user preferences (destination, budget, dates, interests)
2. Summarize preferences using the tool
3. Create recommendations based on the summary
4. Update the summary when preferences change""",
    tools=[summarize_preferences],
)

print("Summarizing travel agent created with context tools")

In [ ]:
# Demonstrate the summarization pattern
summary_session = summarizing_agent.create_session()

# Provide a batch of preferences
response = await summarizing_agent.run(
    "I want to visit Greece. I love seafood, history, and island hopping. "
    "Budget is $4000 for two weeks. Traveling with my partner in June. "
    "Please record these preferences using your summarization tool.",
    session=summary_session,
)
print(f"Agent: {response}\n")

# Ask the agent to use the recorded context
response = await summarizing_agent.run(
    "Now, based on what you've recorded, suggest the top 3 islands we should visit.",
    session=summary_session,
)
print(f"Agent: {response}\n")

## 요약

이번 수업에서는 Microsoft Agent Framework를 사용하여 장시간 실행되는 에이전트 대화에서 컨텍스트를 관리하는 방법을 배웠습니다:

### 주요 개념
- **컨텍스트 창은 유한합니다** — 대화 기록의 모든 토큰은 비용이 발생하며 제한에 포함됩니다.
- <strong>요약 도구</strong>는 누적된 컨텍스트를 간결한 요약으로 압축하여 토큰 사용량을 줄이면서 필수 정보를 보존합니다.
- <strong>에이전트 스크래치패드</strong>는 대화 축소에도 불구하고 지속되는 외부 메모리를 제공합니다.

### 구축한 내용
- 다중 턴 대화에서 연속성을 유지하는 **컨텍스트 인식 에이전트**
- 핵심 사용자 정보를 간결하게 기록하는 **요약 도구**(`summarize_preferences`)
- 컨텍스트 유지 및 변경 처리 예를 보여주는 **다중 턴 대화**

### 실제 활용 사례
- **고객 지원 봇**: 장시간 지원 세션 동안 선호도 기억
- **개인 비서**: 컨텍스트를 다시 설명하지 않고 진행 중인 프로젝트 추적
- **교육 튜터**: 여러 상호작용에서 학생 진행 상황 유지

### 다음 단계
- 파일 기반 지속성을 가진 완전한 스크래치패드 도구 구현
- 요약 후 자동 기록 축소 기능 추가
- 의미 기억 검색을 위한 벡터 데이터베이스와 결합
- 전체 컨텍스트로 며칠 후에도 대화를 다시 시작할 수 있는 에이전트 구축


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**면책 조항**:
이 문서는 AI 번역 서비스 [Co-op Translator](https://github.com/Azure/co-op-translator)를 사용하여 번역되었습니다. 정확성을 기하기 위해 노력하고 있으나, 자동 번역은 오류나 부정확한 부분이 있을 수 있음을 유의하시기 바랍니다. 원본 문서의 원어본이 권위 있는 자료로 간주되어야 합니다. 중요한 정보의 경우, 전문가의 인간 번역을 권장합니다. 이 번역 사용으로 인해 발생하는 오해나 잘못된 해석에 대해 당사는 책임을 지지 않습니다.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
